# Ejercicio 1 — Carga y armonización de datos

In [1]:
import os
import re
import tempfile
from functools import reduce

import openpyxl
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType,
                               DoubleType, IntegerType, LongType)

spark = (SparkSession.builder
         .master("local[*]")
         .appName("Lab7")
         .config("spark.driver.memory", "4g")
         .getOrCreate())

RAW_DIR = "../working_dir/raw"
PARQUET_DIR = "../working_dir/parquet"
# Parquet intermedio de cada archivo: va a una carpeta temporal fuera del proyecto
# (en la carpeta compartida con Docker, Spark a veces no puede sobrescribirlo)
STAGING_DIR = os.path.join(tempfile.gettempdir(), "lab7_staging")

print(spark.version)
print(sorted(os.listdir(RAW_DIR)))

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/24 22:35:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


3.5.1
['Base-de-datos-Personas-ENEIC-I-2026.xlsx', 'Base-de-datos-Personas-ENEIC-III-2025.xlsx', 'Base-de-datos-Personas-ENEIC-IV-2025.xlsx', 'Personas-ENEIC-T2-2025.xlsx', 'Personas_ENEIC_T1_2025.xlsx']


## 1.1 Identificación del período de cada archivo

In [2]:
# Metadatos de cada archivo: el período sale del archivo, no de la columna TRIMESTRE
ARCHIVOS = [
    {"archivo": "Personas_ENEIC_T1_2025.xlsx",                "periodo": "2025T1", "anio": 2025, "trimestre": 1},
    {"archivo": "Personas-ENEIC-T2-2025.xlsx",                "periodo": "2025T2", "anio": 2025, "trimestre": 2},
    {"archivo": "Base-de-datos-Personas-ENEIC-III-2025.xlsx", "periodo": "2025T3", "anio": 2025, "trimestre": 3},
    {"archivo": "Base-de-datos-Personas-ENEIC-IV-2025.xlsx",  "periodo": "2025T4", "anio": 2025, "trimestre": 4},
    {"archivo": "Base-de-datos-Personas-ENEIC-I-2026.xlsx",   "periodo": "2026T1", "anio": 2026, "trimestre": 1},
]
pd.DataFrame(ARCHIVOS)

,archivo,periodo,anio,trimestre
0,Personas_ENEIC_T1_2025.xlsx,2025T1,2025,1
1,Personas-ENEIC-T2-2025.xlsx,2025T2,2025,2
2,Base-de-datos-Personas-ENEIC-III-2025.xlsx,2025T3,2025,3
3,Base-de-datos-Personas-ENEIC-IV-2025.xlsx,2025T4,2025,4
4,Base-de-datos-Personas-ENEIC-I-2026.xlsx,2026T1,2026,1


## 1.2 Selección de columnas y tipos

In [3]:
# Columna original -> nombre analítico
COLUMNAS = {
    "P05D01":      "salario_mensual",
    "P02A03":      "edad",
    "P05C07A":     "antiguedad_anios",
    "P05C07B":     "antiguedad_meses",
    "P05H01A":     "horas_semanales",
    "P03A03A":     "nivel_educativo",
    "P05C16":      "categoria_ocupacional",
    "DOMINIO":     "dominio",
    "OCUPADOS":    "ocupado",
    "NUM_HOGAR":   "NUM_HOGAR",
    "NUM_PERSONA": "NUM_PERSONA",
    "FACTOR":      "FACTOR",
    "ANIO":        "ANIO",
    "TRIMESTRE":   "TRIMESTRE",
}

NUMERICAS = ["salario_mensual", "edad", "antiguedad_anios", "antiguedad_meses",
             "horas_semanales", "FACTOR"]
CATEGORICAS = ["nivel_educativo", "categoria_ocupacional", "dominio", "ocupado"]
ENTERAS = {"NUM_HOGAR": LongType(), "NUM_PERSONA": IntegerType(),
           "ANIO": IntegerType(), "TRIMESTRE": IntegerType()}

ESQUEMA = StructType(
    [StructField("archivo_origen", StringType(), False),
     StructField("periodo_archivo", StringType(), False),
     StructField("anio_archivo", IntegerType(), False),
     StructField("trimestre_calendario", IntegerType(), False)]
    + [StructField(c, ENTERAS[c], True) for c in ENTERAS]
    + [StructField(c, StringType(), True) for c in CATEGORICAS]
    + [StructField(c, DoubleType(), True) for c in NUMERICAS]
)
ORDEN_COLUMNAS = [f.name for f in ESQUEMA.fields]

## 1.3 Homologación de códigos

In [4]:
def normalizar_codigo(valor):
    """Convierte un código a texto sin decimales ('5', 5, 5.0 -> '5'). Vacíos -> None."""
    if valor is None or (isinstance(valor, float) and pd.isna(valor)):
        return None
    if isinstance(valor, (int, float)):
        return str(int(valor)) if float(valor).is_integer() else str(valor)
    texto = str(valor).strip()
    if texto == "":
        return None
    if re.fullmatch(r"-?\d+\.0+", texto):   # '5.0' -> '5'
        texto = texto.split(".")[0]
    return texto  # códigos no numéricos se conservan para validarlos contra el diccionario


def a_numero(serie):
    """Convierte a float; lo que no sea numérico queda como NaN (se contabiliza después)."""
    return pd.to_numeric(serie.map(lambda v: v.strip() if isinstance(v, str) else v),
                         errors="coerce")


def a_valor_spark(v):
    """pandas usa NaN para faltantes; Spark necesita None para que queden como null."""
    if v is None or (isinstance(v, float) and pd.isna(v)) or v is pd.NA:
        return None
    return v

## 1.4 Carga individual de cada archivo

In [5]:
def contar_columnas_originales(ruta):
    wb = openpyxl.load_workbook(ruta, read_only=True)
    ws = wb[wb.sheetnames[0]]
    n = len(next(ws.iter_rows(min_row=1, max_row=1, values_only=True)))
    wb.close()
    return n


def cargar_archivo(meta):
    ruta = os.path.join(RAW_DIR, meta["archivo"])
    pdf = pd.read_excel(ruta, usecols=list(COLUMNAS), dtype=object)
    pdf = pdf.rename(columns=COLUMNAS)

    for c in NUMERICAS:
        pdf[c] = a_numero(pdf[c]).astype(float).astype(object)
    for c in CATEGORICAS:
        pdf[c] = pdf[c].map(normalizar_codigo)
    for c in ENTERAS:
        pdf[c] = a_numero(pdf[c]).map(lambda v: None if pd.isna(v) else int(v))

    pdf["archivo_origen"] = meta["archivo"]
    pdf["periodo_archivo"] = meta["periodo"]
    pdf["anio_archivo"] = meta["anio"]
    pdf["trimestre_calendario"] = meta["trimestre"]

    filas = [[a_valor_spark(v) for v in fila]
             for fila in pdf[ORDEN_COLUMNAS].itertuples(index=False, name=None)]
    sdf = spark.createDataFrame(filas, schema=ESQUEMA)

    salida = f"{STAGING_DIR}/{meta['periodo']}"
    sdf.write.mode("overwrite").parquet(salida)
    del pdf, filas
    return spark.read.parquet(salida)


resumen_carga = []
bases = {}
for meta in ARCHIVOS:
    ruta = os.path.join(RAW_DIR, meta["archivo"])
    n_cols = contar_columnas_originales(ruta)
    sdf = cargar_archivo(meta)
    bases[meta["periodo"]] = sdf
    resumen_carga.append({"periodo_archivo": meta["periodo"], "archivo_origen": meta["archivo"],
                          "columnas_originales": n_cols, "registros_originales": sdf.count()})
    print(f"{meta['periodo']}: listo")

pd.DataFrame(resumen_carga)

2025T1: listo
2025T2: listo
2025T3: listo
2025T4: listo
2026T1: listo


,periodo_archivo,archivo_origen,columnas_originales,registros_originales
0,2025T1,Personas_ENEIC_T1_2025.xlsx,270,51588
1,2025T2,Personas-ENEIC-T2-2025.xlsx,270,51167
2,2025T3,Base-de-datos-Personas-ENEIC-III-2025.xlsx,270,51583
3,2025T4,Base-de-datos-Personas-ENEIC-IV-2025.xlsx,302,49338
4,2026T1,Base-de-datos-Personas-ENEIC-I-2026.xlsx,270,49843


## 1.5 Unión de los archivos de 2025

In [6]:
periodos_2025 = ["2025T1", "2025T2", "2025T3", "2025T4"]
df_2025 = reduce(lambda a, b: a.unionByName(b), [bases[p] for p in periodos_2025])
df_2026 = bases["2026T1"]

print("Registros 2025:", df_2025.count())
print("Registros 2026:", df_2026.count())
df_2025.printSchema()

Registros 2025: 203676
Registros 2026: 49843
root
 |-- archivo_origen: string (nullable = true)
 |-- periodo_archivo: string (nullable = true)
 |-- anio_archivo: integer (nullable = true)
 |-- trimestre_calendario: integer (nullable = true)
 |-- NUM_HOGAR: long (nullable = true)
 |-- NUM_PERSONA: integer (nullable = true)
 |-- ANIO: integer (nullable = true)
 |-- TRIMESTRE: integer (nullable = true)
 |-- nivel_educativo: string (nullable = true)
 |-- categoria_ocupacional: string (nullable = true)
 |-- dominio: string (nullable = true)
 |-- ocupado: string (nullable = true)
 |-- salario_mensual: double (nullable = true)
 |-- edad: double (nullable = true)
 |-- antiguedad_anios: double (nullable = true)
 |-- antiguedad_meses: double (nullable = true)
 |-- horas_semanales: double (nullable = true)
 |-- FACTOR: double (nullable = true)



### Verificación de la procedencia y de la columna `TRIMESTRE`

In [7]:
(df_2025.unionByName(df_2026)
    .groupBy("periodo_archivo", "anio_archivo", "trimestre_calendario", "ANIO", "TRIMESTRE")
    .count()
    .orderBy("periodo_archivo", "TRIMESTRE")
    .show(truncate=False))

+---------------+------------+--------------------+----+---------+-----+
|periodo_archivo|anio_archivo|trimestre_calendario|ANIO|TRIMESTRE|count|
+---------------+------------+--------------------+----+---------+-----+
|2025T1         |2025        |1                   |2025|2        |51588|
|2025T2         |2025        |2                   |2025|2        |175  |
|2025T2         |2025        |2                   |2025|3        |50992|
|2025T3         |2025        |3                   |2025|4        |51583|
|2025T4         |2025        |4                   |2025|5        |49338|
|2026T1         |2026        |1                   |2026|6        |49843|
+---------------+------------+--------------------+----+---------+-----+



### Verificación de la homologación de códigos

Después de normalizar, los códigos categóricos tienen la misma representación en todos los archivos, incluido II de 2025, que venía como texto.

In [8]:
for c in ["categoria_ocupacional", "dominio", "ocupado"]:
    print(f"== {c} ==")
    (df_2025.unionByName(df_2026)
        .groupBy("periodo_archivo").pivot(c).count()
        .orderBy("periodo_archivo")
        .show(truncate=False))

== categoria_ocupacional ==
+---------------+-----+----+----+----+----+----+---+----+---+----+
|periodo_archivo|null |1   |2   |3   |4   |5   |6  |7   |8  |9   |
+---------------+-----+----+----+----+----+----+---+----+---+----+
|2025T1         |29315|1623|6825|4057|914 |5362|497|1625|101|1269|
|2025T2         |28824|1634|7136|3702|1020|5245|649|1713|84 |1160|
|2025T3         |29210|1681|7566|3212|991 |5320|709|1736|123|1035|
|2025T4         |28105|1637|7175|2871|981 |5024|769|1672|110|994 |
|2026T1         |28109|1646|7503|3093|1016|5066|783|1554|112|961 |
+---------------+-----+----+----+----+----+----+---+----+---+----+

== dominio ==
+---------------+-----+-----+-----+
|periodo_archivo|1    |2    |3    |
+---------------+-----+-----+-----+
|2025T1         |16988|19913|14687|
|2025T2         |16928|19821|14418|
|2025T3         |17254|19906|14423|
|2025T4         |16351|18930|14057|
|2026T1         |16905|18952|13986|
+---------------+-----+-----+-----+

== ocupado ==
+--------------

### Esquema y cinco registros de las columnas seleccionadas

In [9]:
df_2025.show(5, truncate=False)

+---------------------------+---------------+------------+--------------------+---------+-----------+----+---------+---------------+---------------------+-------+-------+---------------+----+----------------+----------------+---------------+------+
|archivo_origen             |periodo_archivo|anio_archivo|trimestre_calendario|NUM_HOGAR|NUM_PERSONA|ANIO|TRIMESTRE|nivel_educativo|categoria_ocupacional|dominio|ocupado|salario_mensual|edad|antiguedad_anios|antiguedad_meses|horas_semanales|FACTOR|
+---------------------------+---------------+------------+--------------------+---------+-----------+----+---------+---------------+---------------------+-------+-------+---------------+----+----------------+----------------+---------------+------+
|Personas_ENEIC_T1_2025.xlsx|2025T1         |2025        |1                   |1945     |2          |2025|2        |0              |5                    |1      |1      |NULL           |34.0|2.0             |0.0             |91.0           |47.0  |
|Per

## 1.6 Faltantes por variable (antes de los filtros)

Se cuentan los valores nulos de cada variable seleccionada sobre **todos** los registros cargados, antes de aplicar cualquier filtro. En las variables numéricas también se cuentan como faltantes los `NaN`.

In [10]:
VARS_SELECCIONADAS = list(COLUMNAS.values())

def tabla_faltantes(df, conjunto):
    total = df.count()
    exprs = []
    for c in VARS_SELECCIONADAS:
        nulo = F.col(c).isNull()
        if c in NUMERICAS:
            nulo = nulo | F.isnan(c)
        exprs.append(F.sum(nulo.cast("int")).alias(c))
    fila = df.agg(*exprs).first().asDict()
    t = pd.DataFrame({"variable": list(fila), f"faltantes_{conjunto}": list(fila.values())})
    t[f"pct_{conjunto}"] = (100 * t[f"faltantes_{conjunto}"] / total).round(2)
    return t

faltantes = tabla_faltantes(df_2025, "2025").merge(tabla_faltantes(df_2026, "2026"), on="variable")
faltantes

,variable,faltantes_2025,pct_2025,faltantes_2026,pct_2026
0,salario_mensual,150651,73.97,36585,73.40
1,edad,0,0.00,0,0.00
2,antiguedad_anios,115454,56.69,28109,56.40
3,antiguedad_meses,115454,56.69,28109,56.40
4,horas_semanales,115454,56.69,28109,56.40
5,nivel_educativo,26344,12.93,6011,12.06
6,categoria_ocupacional,115454,56.69,28109,56.40
7,dominio,0,0.00,0,0.00
8,ocupado,115454,56.69,28109,56.40
9,NUM_HOGAR,0,0.00,0,0.00


### ¿Faltante o "no aplica"?

Muchas preguntas de la encuesta solo se hacen a una parte de las personas, así que un nulo puede significar que **la pregunta no le correspondía** a esa persona, y no que falte la respuesta. Las tablas siguientes revisan si los nulos coinciden con esos saltos del cuestionario.

In [11]:
df_todo = df_2025.unionByName(df_2026)

def nulos(c):
    return F.sum(F.col(c).isNull().cast("int"))

print("Nivel educativo nulo según edad")
(df_todo
    .groupBy(F.when(F.col("edad") < 7, "0 a 6 años").otherwise("7 años o más").alias("grupo_edad"))
    .agg(F.count("*").alias("registros"), nulos("nivel_educativo").alias("educacion_nula"))
    .orderBy("grupo_edad").show())

print("Variables laborales nulas según condición de ocupación")
(df_todo
    .groupBy(F.when(F.col("ocupado") == "1", "ocupado").otherwise("no ocupado (OCUPADOS nulo)").alias("condicion"))
    .agg(F.count("*").alias("registros"),
         F.min("edad").alias("edad_min"),
         nulos("categoria_ocupacional").alias("categoria_nula"),
         nulos("antiguedad_anios").alias("antig_anios_nula"),
         nulos("antiguedad_meses").alias("antig_meses_nula"),
         nulos("horas_semanales").alias("horas_nulas"),
         nulos("salario_mensual").alias("salario_nulo"))
    .show(truncate=False))

print("Salario nulo entre ocupados, según si son asalariados (P05C16 en 1-4)")
(df_todo.filter(F.col("ocupado") == "1")
    .groupBy(F.col("categoria_ocupacional").isin("1", "2", "3", "4").alias("asalariado"))
    .agg(F.count("*").alias("registros"), nulos("salario_mensual").alias("salario_nulo"))
    .show())

Nivel educativo nulo según edad
+------------+---------+--------------+
|  grupo_edad|registros|educacion_nula|
+------------+---------+--------------+
|  0 a 6 años|    32355|         32355|
|7 años o más|   221164|             0|
+------------+---------+--------------+

Variables laborales nulas según condición de ocupación
+--------------------------+---------+--------+--------------+----------------+----------------+-----------+------------+
|condicion                 |registros|edad_min|categoria_nula|antig_anios_nula|antig_meses_nula|horas_nulas|salario_nulo|
+--------------------------+---------+--------+--------------+----------------+----------------+-----------+------------+
|ocupado                   |109956   |15.0    |0             |0               |0               |0          |43673       |
|no ocupado (OCUPADOS nulo)|143563   |0.0     |143563        |143563          |143563          |143563     |143563      |
+--------------------------+---------+--------+--------------+

**Interpretación.**

- Los identificadores (`NUM_HOGAR`, `NUM_PERSONA`), `FACTOR`, `ANIO`, `TRIMESTRE`, `edad` y `dominio` están completos.
- `nivel_educativo` es nulo en ≈ 12–13 % de los registros, y **todos** esos casos son personas de 0 a 6 años: la pregunta de educación solo se hace a partir de los 7 años.
- `ocupado`, `categoria_ocupacional`, `antiguedad_*` y `horas_semanales` son nulos en ≈ 56–57 % de los registros, **exactamente** los mismos: las personas no ocupadas, a quienes no se les hace la sección de empleo. Tampoco hay personas ocupadas menores de 15 años.
- `salario_mensual` es la variable con más nulos (≈ 74 %). Esos casos son las personas no ocupadas y, entre las ocupadas, **solo** las que no son asalariadas (cuenta propia, patronos, no remunerados). Todos los asalariados tienen salario registrado.

Es decir, en las variables seleccionadas **todos los faltantes son estructurales** ("no aplica") y no hay respuestas perdidas dentro de la población a la que sí le correspondía la pregunta. Por eso casi todas las exclusiones de la sección de filtros vienen de definir la población y no de problemas de calidad.

## 1.7 Unicidad de la clave `periodo_archivo` + `NUM_HOGAR` + `NUM_PERSONA`

Dentro de un mismo período, cada persona debería aparecer una sola vez. Si hubiera claves repetidas, se revisaría si son **repeticiones exactas** (la fila completa es igual) o **registros en conflicto** (misma clave, valores distintos). No se usa `dropDuplicates()` para ocultar el problema.

In [12]:
CLAVE = ["periodo_archivo", "NUM_HOGAR", "NUM_PERSONA"]

def revisar_unicidad(df, conjunto):
    total = df.count()
    claves_repetidas = df.groupBy(CLAVE).count().filter(F.col("count") > 1)
    n_claves_repetidas = claves_repetidas.count()
    filas_exactas_repetidas = total - df.distinct().count()
    # Claves que siguen repetidas aun quitando filas idénticas -> registros en conflicto
    claves_en_conflicto = df.distinct().groupBy(CLAVE).count().filter(F.col("count") > 1).count()
    return {"conjunto": conjunto, "registros": total,
            "claves_distintas": df.select(CLAVE).distinct().count(),
            "claves_repetidas": n_claves_repetidas,
            "filas_repetidas_exactas": filas_exactas_repetidas,
            "claves_en_conflicto": claves_en_conflicto}

unicidad = pd.DataFrame([revisar_unicidad(df_2025, "2025"), revisar_unicidad(df_2026, "2026")])
unicidad

,conjunto,registros,claves_distintas,claves_repetidas,filas_repetidas_exactas,claves_en_conflicto
0,2025,203676,203676,0,0,0
1,2026,49843,49843,0,0,0


La clave es **única** en los dos conjuntos: el número de claves distintas es igual al número de registros, no hay filas repetidas exactas ni registros en conflicto. No hace falta eliminar nada.

### La misma persona en varios períodos

La unicidad se exige **dentro** de cada período, no entre períodos. La ENEIC tiene un diseño longitudinal con rotación, así que un mismo hogar y persona puede aparecer en varios trimestres:

In [13]:
apariciones = (df_2025
    .groupBy("NUM_HOGAR", "NUM_PERSONA")
    .agg(F.countDistinct("periodo_archivo").alias("periodos_en_que_aparece"))
    .groupBy("periodos_en_que_aparece").count()
    .withColumnRenamed("count", "combinaciones_hogar_persona")
    .orderBy("periodos_en_que_aparece")
    .toPandas())
apariciones

,periodos_en_que_aparece,combinaciones_hogar_persona
0,1,31273
1,2,27771
2,3,16279
3,4,17006


In [14]:
# ¿Es la misma persona? Se compara la edad reportada en dos trimestres consecutivos
t1 = df_2025.filter(F.col("periodo_archivo") == "2025T1").select("NUM_HOGAR", "NUM_PERSONA", F.col("edad").alias("edad_t1"))
t2 = df_2025.filter(F.col("periodo_archivo") == "2025T2").select("NUM_HOGAR", "NUM_PERSONA", F.col("edad").alias("edad_t2"))
(t1.join(t2, ["NUM_HOGAR", "NUM_PERSONA"])
    .withColumn("diferencia_edad", F.col("edad_t2") - F.col("edad_t1"))
    .groupBy(F.when(F.col("diferencia_edad").between(0, 1), "0 o 1 año").otherwise("otra").alias("diferencia_edad"))
    .count().show())

+---------------+-----+
|diferencia_edad|count|
+---------------+-----+
|           otra|  116|
|      0 o 1 año|36892|
+---------------+-----+



Más de la mitad de las combinaciones hogar–persona de 2025 aparecen en dos o más trimestres, y entre T1 y T2 casi todas reportan la misma edad o un año más, lo que indica que efectivamente se trata de la misma persona observada otra vez. Estas filas **no son duplicados**: son observaciones distintas de la misma persona en períodos distintos. Por eso el número de filas de 2025 no equivale al número de personas distintas.

## 1.8 Validación de códigos contra el diccionario

Los códigos válidos de cada variable categórica se leen de la sección *Valores de variable* de los diccionarios. Primero se revisa que los cinco diccionarios definan los mismos códigos y después se clasifica cada valor de los datos como **válido**, **nulo** o **no reconocido**.

In [15]:
DICC_DIR = "../working_dir/diccionarios"
DICCIONARIOS = {
    "2025T1": "Diccionario_Personas_ENEIC_I-2025.xlsx",
    "2025T2": "Diccionario_Personas_ENEIC_II-2025.xlsx",
    "2025T3": "Diccionario-Personas-ENEIC-III-2025.xlsx",
    "2025T4": "Diccionario-Personas-ENEIC-IV-2025.xlsx",
    "2026T1": "Diccionario-Personas-ENEIC-I-2026.xlsx",
}
# Variable original -> nombre analítico
VARS_DICC = {"P03A03A": "nivel_educativo", "P05C16": "categoria_ocupacional",
             "DOMINIO": "dominio", "OCUPADOS": "ocupado"}


def leer_valores_diccionario(ruta):
    d = pd.read_excel(ruta, header=None, dtype=str)
    inicio = d.index[d[0].str.strip().eq("Valores de variable")][0]
    valores = d.loc[inicio + 2:, [0, 1, 2]]
    valores.columns = ["variable", "codigo", "etiqueta"]
    valores["variable"] = valores["variable"].ffill().str.strip()
    valores = valores.dropna(subset=["codigo"])
    valores["codigo"] = valores["codigo"].map(normalizar_codigo)
    valores["etiqueta"] = valores["etiqueta"].str.strip(" ¿?")
    return valores[valores["variable"].isin(VARS_DICC)]


valores_dicc = pd.concat(
    [leer_valores_diccionario(os.path.join(DICC_DIR, f)).assign(periodo=p) for p, f in DICCIONARIOS.items()],
    ignore_index=True)

# ¿Los cinco diccionarios definen los mismos códigos y etiquetas?
consistencia = (valores_dicc.groupby(["variable", "codigo"])
                .agg(diccionarios=("periodo", "nunique"), etiquetas_distintas=("etiqueta", "nunique"),
                     etiqueta=("etiqueta", "first"))
                .reset_index())
consistencia

,variable,codigo,diccionarios,etiquetas_distintas,etiqueta
0,DOMINIO,1,5,1,Urbano Metropolitano
1,DOMINIO,2,5,1,Resto Urbano
2,DOMINIO,3,5,1,Rural Nacional
3,OCUPADOS,1,5,1,Población ocupada
4,P03A03A,0,5,1,NINGUNO
5,P03A03A,1,5,1,PREPRIMARIA
6,P03A03A,2,5,1,PRIMARIA
7,P03A03A,3,5,1,BÁSICO
8,P03A03A,4,5,1,DIVERSIFICADO
9,P03A03A,5,5,1,SUPERIOR


In [16]:
CODIGOS_VALIDOS = {VARS_DICC[v]: sorted(g["codigo"].unique())
                   for v, g in valores_dicc.groupby("variable")}
ETIQUETAS = {VARS_DICC[v]: dict(zip(g["codigo"], g["etiqueta"]))
             for v, g in valores_dicc.drop_duplicates(["variable", "codigo"]).groupby("variable")}

validacion = []
for col, validos in CODIGOS_VALIDOS.items():
    estado = (F.when(F.col(col).isNull(), "nulo")
               .when(F.col(col).isin(validos), "válido")
               .otherwise("no reconocido"))
    validacion.append(df_todo.groupBy("periodo_archivo", estado.alias("estado")).count()
                      .withColumn("variable", F.lit(col)).toPandas())

(pd.concat(validacion)
   .pivot_table(index=["variable", "periodo_archivo"], columns="estado", values="count", fill_value=0)
   .astype(int))

estado                                  nulo  válido
variable              periodo_archivo               
categoria_ocupacional 2025T1           29315   22273
                      2025T2           28824   22343
                      2025T3           29210   22373
                      2025T4           28105   21233
                      2026T1           28109   21734
dominio               2025T1               0   51588
                      2025T2               0   51167
                      2025T3               0   51583
                      2025T4               0   49338
                      2026T1               0   49843
nivel_educativo       2025T1            6928   44660
                      2025T2            6678   44489
                      2025T3            6548   45035
                      2025T4            6190   43148
                      2026T1            6011   43832
ocupado               2025T1           29315   22273
                      2025T2           28824   22343
                      2025T3           29210   22373
                      2025T4           28105   21233
                      2026T1           28109   21734

**Interpretación.** Los cinco diccionarios definen los mismos códigos con las mismas etiquetas: `P03A03A` de 0 (NINGUNO) a 7 (DOCTORADO), `P05C16` de 1 a 9, `DOMINIO` de 1 a 3 y `OCUPADOS` = 1. En los datos **no hay códigos no reconocidos**: todos los valores son válidos o nulos, y los nulos son los estructurales que se explicaron en la sección 1.6.

Aun así, después de los filtros las variables categóricas se recodifican para que cualquier valor nulo o fuera del diccionario quede como `DESCONOCIDO` (sección 1.11). El código educativo `0` es una categoría válida ("ninguno") y **no** se trata como faltante.

## 1.9 Construcción de la antigüedad

In [17]:
def agregar_antiguedad(df):
    return df.withColumn("antiguedad",
                         F.col("antiguedad_anios") + F.col("antiguedad_meses") / F.lit(12.0))

df_2025 = agregar_antiguedad(df_2025).persist()
df_2026 = agregar_antiguedad(df_2026).persist()

## 1.10 Filtros de población y de calidad

In [18]:
def es_finito(nombre):
    c = F.col(nombre)
    return c.isNotNull() & ~F.isnan(c) & (F.abs(c) != float("inf"))

# (paso, descripción, condición para poder evaluar, condición que debe cumplir)
PASOS_FILTRO = [
    (1, "Edad finita y >= 15",
        es_finito("edad"),
        F.col("edad") >= 15),
    (2, "Ocupado (OCUPADOS = 1)",
        F.col("ocupado").isNotNull(),
        F.col("ocupado") == "1"),
    (3, "Asalariado (P05C16 en 1-4)",
        F.col("categoria_ocupacional").isNotNull(),
        F.col("categoria_ocupacional").isin("1", "2", "3", "4")),
    (4, "Salario finito y > 0",
        es_finito("salario_mensual"),
        F.col("salario_mensual") > 0),
    (5, "Antigüedad en años >= 0",
        es_finito("antiguedad_anios"),
        F.col("antiguedad_anios") >= 0),
    (6, "Meses enteros entre 0 y 11",
        es_finito("antiguedad_meses"),
        (F.col("antiguedad_meses") == F.floor("antiguedad_meses"))
        & F.col("antiguedad_meses").between(0, 11)),
    (7, "Antigüedad <= edad",
        es_finito("antiguedad") & es_finito("edad"),
        F.col("antiguedad") <= F.col("edad")),
    (8, "Horas > 0 y <= 168",
        es_finito("horas_semanales"),
        (F.col("horas_semanales") > 0) & (F.col("horas_semanales") <= 168)),
]


def aplicar_filtros(df):
    """Aplica los pasos en orden y devuelve (df filtrado, bitácora por período y paso)."""
    bitacora = []
    actual = df
    for paso, descripcion, evaluable, cumple in PASOS_FILTRO:
        conteo = (actual
                  .withColumn("_evaluable", evaluable)
                  .withColumn("_cumple", F.col("_evaluable") & F.coalesce(cumple, F.lit(False)))
                  .groupBy("periodo_archivo")
                  .agg(F.count("*").alias("entrada"),
                       F.sum((~F.col("_evaluable")).cast("int")).alias("excluidos_no_evaluable"),
                       F.sum((F.col("_evaluable") & ~F.col("_cumple")).cast("int")).alias("excluidos_no_cumple"),
                       F.sum(F.col("_cumple").cast("int")).alias("restantes"))
                  .toPandas())
        conteo.insert(0, "criterio", descripcion)
        conteo.insert(0, "paso", paso)
        bitacora.append(conteo)
        actual = actual.filter(evaluable & cumple)
    bitacora = pd.concat(bitacora, ignore_index=True)
    bitacora["excluidos_total"] = bitacora["excluidos_no_evaluable"] + bitacora["excluidos_no_cumple"]
    return actual, bitacora.sort_values(["periodo_archivo", "paso"]).reset_index(drop=True)


df_2025_prep, bitacora_2025 = aplicar_filtros(df_2025)
df_2026_prep, bitacora_2026 = aplicar_filtros(df_2026)
df_2025_prep = df_2025_prep.persist()
df_2026_prep = df_2026_prep.persist()
bitacora = pd.concat([bitacora_2025, bitacora_2026], ignore_index=True)

26/09/24 22:38:11 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


### Registros excluidos en cada paso

In [19]:
bitacora["conjunto"] = bitacora["periodo_archivo"].str[:4]
resumen_pasos = (bitacora
    .groupby(["conjunto", "paso", "criterio"], as_index=False)
    [["entrada", "excluidos_no_evaluable", "excluidos_no_cumple", "excluidos_total", "restantes"]]
    .sum())
resumen_pasos["pct_excluido_del_paso"] = (100 * resumen_pasos["excluidos_total"]
                                          / resumen_pasos["entrada"]).round(2)
resumen_pasos

,conjunto,paso,criterio,entrada,excluidos_no_evaluable,excluidos_no_cumple,excluidos_total,restantes,pct_excluido_del_paso
0,2025,1,Edad finita y >= 15,203676,0,62886,62886,140790,30.88
1,2025,2,Ocupado (OCUPADOS = 1),140790,52568,0,52568,88222,37.34
2,2025,3,Asalariado (P05C16 en 1-4),88222,0,35197,35197,53025,39.90
3,2025,4,Salario finito y > 0,53025,0,0,0,53025,0.00
4,2025,5,Antigüedad en años >= 0,53025,0,0,0,53025,0.00
5,2025,6,Meses enteros entre 0 y 11,53025,0,0,0,53025,0.00
6,2025,7,Antigüedad <= edad,53025,0,0,0,53025,0.00
7,2025,8,Horas > 0 y <= 168,53025,0,0,0,53025,0.00
8,2026,1,Edad finita y >= 15,49843,0,14803,14803,35040,29.70
9,2026,2,Ocupado (OCUPADOS = 1),35040,13306,0,13306,21734,37.97


Detalle de registros excluidos por archivo en cada paso:

In [20]:
bitacora.pivot_table(index=["paso", "criterio"], columns="periodo_archivo",
                     values="excluidos_total", aggfunc="sum")

,periodo_archivo,2025T1,2025T2,2025T3,2025T4,2026T1
paso,criterio,,,,,
1,Edad finita y >= 15,16253,15882,15767,14984,14803
2,Ocupado (OCUPADOS = 1),13062,12942,13443,13121,13306
3,Asalariado (P05C16 en 1-4),8854,8851,8923,8569,8476
4,Salario finito y > 0,0,0,0,0,0
5,Antigüedad en años >= 0,0,0,0,0,0
6,Meses enteros entre 0 y 11,0,0,0,0,0
7,Antigüedad <= edad,0,0,0,0,0
8,Horas > 0 y <= 168,0,0,0,0,0


### Registros por archivo antes y después de los filtros

In [21]:
antes = pd.DataFrame(resumen_carga)[["periodo_archivo", "registros_originales"]]
despues = (df_2025_prep.unionByName(df_2026_prep)
           .groupBy("periodo_archivo").count()
           .withColumnRenamed("count", "registros_filtrados")
           .toPandas())
antes_despues = antes.merge(despues, on="periodo_archivo", how="left")
antes_despues["excluidos"] = antes_despues["registros_originales"] - antes_despues["registros_filtrados"]
antes_despues["pct_conservado"] = (100 * antes_despues["registros_filtrados"]
                                   / antes_despues["registros_originales"]).round(2)
antes_despues

,periodo_archivo,registros_originales,registros_filtrados,excluidos,pct_conservado
0,2025T1,51588,13419,38169,26.01
1,2025T2,51167,13492,37675,26.37
2,2025T3,51583,13450,38133,26.07
3,2025T4,49338,12664,36674,25.67
4,2026T1,49843,13258,36585,26.60


## 1.11 Recodificación de categorías (`DESCONOCIDO`) y etiquetas

In [22]:
CATEGORICAS_MODELO = ["nivel_educativo", "categoria_ocupacional", "dominio"]

def recodificar_categoricas(df):
    for col in CATEGORICAS_MODELO:
        validos = CODIGOS_VALIDOS[col]
        df = df.withColumn(col, F.when(F.col(col).isin(validos), F.col(col))
                                 .otherwise(F.lit("DESCONOCIDO")))
        mapa = F.create_map(*[F.lit(x) for par in ETIQUETAS[col].items() for x in par])
        df = df.withColumn(f"{col}_etiqueta",
                           F.coalesce(mapa[F.col(col)], F.lit("DESCONOCIDO")))
    return df

df_2025_prep = recodificar_categoricas(df_2025_prep)
df_2026_prep = recodificar_categoricas(df_2026_prep)

for col in CATEGORICAS_MODELO:
    (df_2025_prep.unionByName(df_2026_prep)
        .groupBy(col, f"{col}_etiqueta").pivot("periodo_archivo").count()
        .orderBy(col).show(truncate=False))

+---------------+------------------------+------+------+------+------+------+
|nivel_educativo|nivel_educativo_etiqueta|2025T1|2025T2|2025T3|2025T4|2026T1|
+---------------+------------------------+------+------+------+------+------+
|0              |NINGUNO                 |1050  |1044  |958   |943   |1016  |
|1              |PREPRIMARIA             |138   |122   |108   |91    |80    |
|2              |PRIMARIA                |4062  |4032  |3950  |3778  |3957  |
|3              |BÁSICO                  |2079  |2080  |2095  |1995  |2164  |
|4              |DIVERSIFICADO           |4196  |4334  |4332  |3989  |4117  |
|5              |SUPERIOR                |1674  |1663  |1797  |1684  |1729  |
|6              |MAESTRÍA                |202   |201   |196   |172   |183   |
|7              |DOCTORADO               |18    |16    |14    |12    |12    |
+---------------+------------------------+------+------+------+------+------+

+---------------------+------------------------------+------+--

## 1.12 Guardado de los conjuntos preparados

In [23]:
RUTA_2025 = f"{PARQUET_DIR}/eneic_2025_preparado"
RUTA_2026 = f"{PARQUET_DIR}/eneic_2026_preparado"

df_2025_prep.write.mode("overwrite").parquet(RUTA_2025)
df_2026_prep.write.mode("overwrite").parquet(RUTA_2026)

# Verificación: se vuelve a leer lo guardado
for ruta in [RUTA_2025, RUTA_2026]:
    print(ruta, "->", spark.read.parquet(ruta).count(), "registros")

../working_dir/parquet/eneic_2025_preparado -> 53025 registros
../working_dir/parquet/eneic_2026_preparado -> 13258 registros


## 1.13 Preguntas

In [24]:
def leer_encabezado(ruta):
    wb = openpyxl.load_workbook(ruta, read_only=True)
    encabezado = list(next(wb[wb.sheetnames[0]].iter_rows(min_row=1, max_row=1, values_only=True)))
    wb.close()
    return encabezado

enc = {m["periodo"]: leer_encabezado(os.path.join(RAW_DIR, m["archivo"])) for m in ARCHIVOS}

print("Mismo encabezado (orden incluido) que 2025T3:",
      {p: e == enc["2025T3"] for p, e in enc.items()})
print("Columnas solo en IV-2025:", len(set(enc["2025T4"]) - set(enc["2025T3"])))
print("Columnas de III-2025 que no están en IV-2025:", sorted(set(enc["2025T3"]) - set(enc["2025T4"])))

pd.DataFrame({"variable": list(COLUMNAS),
              "posicion_III_2025": [enc["2025T3"].index(c) + 1 for c in COLUMNAS],
              "posicion_IV_2025": [enc["2025T4"].index(c) + 1 for c in COLUMNAS]})

Mismo encabezado (orden incluido) que 2025T3: {'2025T1': True, '2025T2': True, '2025T3': True, '2025T4': False, '2026T1': True}
Columnas solo en IV-2025: 42
Columnas de III-2025 que no están en IV-2025: ['P02A01C', 'P02A01D', 'P02A01G', 'P02A01I', 'P05C01A', 'P05C02A', 'P05C04A', 'P05G01A', 'P05G02A', 'P05G04A']


,variable,posicion_III_2025,posicion_IV_2025
0,P05D01,105,98
1,P02A03,13,9
2,P05C07A,68,61
3,P05C07B,69,62
4,P05H01A,218,208
5,P03A03A,33,29
6,P05C16,88,81
7,DOMINIO,3,3
8,OCUPADOS,266,298
9,NUM_HOGAR,4,4


**¿Por qué IV de 2025 no puede apilarse por posición de columnas con los otros archivos?**

Porque su estructura es distinta. IV de 2025 tiene 302 columnas contra 270 de los demás: agrega 42 columnas que no existen en los otros archivos (por ejemplo, bloques `P07A…` y `P08A…`) y no incluye 10 que sí están en ellos (como `P02A01C` o `P05C01A`). Por eso las mismas variables quedan en posiciones distintas: `P05D01` (salario) está en la columna 105 en III de 2025 y en la 98 en IV, y `OCUPADOS` pasa de la 266 a la 298. Una unión por posición (`union`) pegaría columnas que no tienen nada que ver, por ejemplo el salario de un archivo con otra pregunta del otro, sin dar ningún error. `unionByName` empareja las columnas por su nombre, así que cada variable queda con su equivalente.

**¿Qué diferencia existe entre un dato ausente porque la pregunta no corresponde y una respuesta no registrada?**

Un dato ausente porque **la pregunta no corresponde** es un salto del cuestionario: a esa persona no se le debía hacer la pregunta, así que el valor no existe. Por ejemplo, la educación no se pregunta a menores de 7 años, y la sección de empleo no se aplica a quienes no están ocupados. Ese nulo no es un error ni un dato perdido, y no tiene sentido imputarlo: es información sobre a quién aplica la variable. Una **respuesta no registrada** es un caso en que la pregunta sí correspondía pero no se obtuvo el dato (la persona no respondió, no sabía o hubo un error de captura). Ese sí es un problema de calidad que puede sesgar los resultados. En las variables de este análisis, todos los nulos resultaron ser del primer tipo (sección 1.6).

**¿Por qué una persona observada en dos períodos no debe eliminarse como duplicado del conjunto longitudinal?**

Porque no es un registro repetido, sino **otra observación** de la misma persona en un momento distinto. En cada trimestre su salario, sus horas, su antigüedad o incluso su ocupación pueden cambiar, y cada observación describe su situación en ese período. La ENEIC está diseñada con rotación para que parte de la muestra se repita entre trimestres. Un duplicado sería la misma persona dos veces **en el mismo período**, y la sección 1.7 muestra que eso no ocurre. Eliminar las repeticiones entre períodos quitaría información válida y haría que los trimestres dejaran de ser comparables.

**¿Por qué el número de registros de la base filtrada no representa a todos los trabajadores del país?**

Por varias razones:
1. **Es una muestra**, no un censo: cada registro representa a muchas personas, y cuántas depende de su `FACTOR` de expansión. Contar filas no es contar trabajadores.
2. **Solo incluye asalariados de 15 años o más con salario positivo**: quedan fuera los trabajadores por cuenta propia, los patronos, los no remunerados y cualquier asalariado sin salario registrado. En Guatemala esos grupos son una parte grande de la fuerza laboral.
3. **Una persona puede aparecer hasta cuatro veces** en la unión de 2025, así que el número de filas no es el número de personas distintas.
4. Los resultados de este laboratorio **no están ponderados**: describen los registros analizados, no la población.

**¿Para qué se usaría `FACTOR`?**

`FACTOR` es el factor de expansión: indica a cuántas personas de la población representa cada registro según el diseño muestral. En un análisis poblacional se usaría como **peso** para estimar totales (por ejemplo, cuántos asalariados hay en el país), medias y medianas ponderadas del salario, y proporciones por dominio o nivel educativo que sí representen a Guatemala. En este laboratorio se conserva para documentar el diseño, pero no se usa como predictor ni para ponderar el clustering, los modelos o las métricas.